In [5]:
import ast
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV

In [6]:
df = pd.read_csv('../court_cases.csv', parse_dates=['Date'])
df.head()

,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,Combined_Issue,Combined_Rule,Combined_Application,plaintiff_label,defendant_label
0,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Holdings Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...","['Under employment and company law, the obliga...",['ABB Holdings Pte Ltd issued and administered...,Claim Dismissed,Liable
1,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Installation Materials (East Asia) Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,['Company officers and senior managers must av...,['ABB Installation Materials (East Asia) Pte L...,Claim Dismissed,Liable
2,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Industry Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",['Senior managers responsible for a business a...,['While serving as General Manager and Vice-Pr...,Claim Allowed,Liable
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,George Wei JC,George Wei,2014-02-14,High Court,Airtrust (Singapore) Pte Ltd (AT) – Suit No 47...,Kao Chai-Chau Linda,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",['Where multiple persons are each legally resp...,"['Airtrust (Singapore) Pte Ltd, through the de...",Claim Allowed,Liable
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,George Wei JC,George Wei,2014-02-14,High Court,Airtrust (Singapore) Pte Ltd (AT) – Suit No 47...,Estate of Peter Fong,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",['Where multiple persons are each legally resp...,"['Airtrust (Singapore) Pte Ltd, through the de...",Claim Allowed,Liable


In [7]:
df = df.drop(columns=['Coram', 'Judge', 'Date', 'Plaintiff_Name', 'Defendant_Name', 'Combined_Rule', 'Combined_Application', 'plaintiff_label'])

In [8]:
df.head()

,Case_Number,Tribunal_Court,Combined_Facts,Combined_Issue,defendant_label
0,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...",Liable
1,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,Liable
2,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",Liable
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
LEGALBERT_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN        = 512          # LegalBERT max token length
TRIBUNAL_DIM   = 16
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. TEXT EXTRACTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def parse_if_string(obj):
    if isinstance(obj, str):
        try:
            return ast.literal_eval(obj)
        except Exception:
            return obj
    return obj

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_facts(raw):
    """
    Flattens Combined_Facts (LLD: List of Lists of Dicts)
    into a single string, preserving Fact_Type and Fact_Date as text.
    """
    raw = parse_if_string(raw)
    if isinstance(raw, list) and raw and isinstance(raw[0], list):
        flat = [item for sublist in raw for item in sublist]
    else:
        flat = raw if isinstance(raw, list) else [raw]

    parts = []
    for item in flat:
        item = parse_if_string(item)
        if not isinstance(item, dict):
            continue
        for v in item.values():
            if isinstance(v, str):
                parts.append(clean(v))
    return " ".join(parts)

def extract_issue(raw):
    """Joins Combined_Issue (LoS: List of Strings) into a single string."""
    raw = parse_if_string(raw)
    if isinstance(raw, list):
        return " ".join(clean(s) for s in raw if str(s).strip())
    return clean(str(raw))

df["facts_text"] = df["Combined_Facts"].apply(extract_facts)
df["issue_text"]  = df["Combined_Issue"].apply(extract_issue)

print(df["facts_text"].iloc[0][:300])
print(df["issue_text"].iloc[0][:300])

party info 1990 01 01 abb holdings pte ltd is part of the abb group a worldwide group of companies that manufacture market and sell several products for industrial and commercial uses including low medium and high voltage circuit breakers party info 1990 01 01 abb holdings pte ltd is part of the sin
whether based on the pre 2003 employment contracts and employee handbook the scope of contractual and equitable obligations of senior management personnel extended to restricting their participation in external business ventures and competitive activities involving medium voltage switchgear products


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. TRIBUNAL_COURT  →  16-dim one-hot vector
# ─────────────────────────────────────────────────────────────────────────────

tribunal_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
tribunal_enc.fit(df[["Tribunal_Court"]])

def get_tribunal_vec(court_val):
    vec = tribunal_enc.transform(
        pd.DataFrame([[court_val]], columns=["Tribunal_Court"])
    )[0]
    if len(vec) < TRIBUNAL_DIM:
        vec = np.pad(vec, (0, TRIBUNAL_DIM - len(vec)))
    else:
        vec = vec[:TRIBUNAL_DIM]
    return vec   # (16,)

tribunal_vecs = np.vstack([get_tribunal_vec(v) for v in df["Tribunal_Court"]])
print(f"Tribunal matrix: {tribunal_vecs.shape}")   # (n_cases, 16)


Tribunal matrix: (424, 16)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD LEGALBERT  (frozen)
# ─────────────────────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(LEGALBERT_NAME)
legalbert = AutoModel.from_pretrained(LEGALBERT_NAME).to(DEVICE)

for param in legalbert.parameters():
    param.requires_grad = False

legalbert.eval()
print(f"LegalBERT loaded on {DEVICE}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25968.09it/s]
BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LegalBERT loaded on cpu


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. LEGALBERT EMBEDDINGS  (batch inference — mean pooling)
#    Embeds facts and issue SEPARATELY to avoid truncation of issue text
# ─────────────────────────────────────────────────────────────────────────────

def get_mean_embeddings(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            max_length=MAX_LEN,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            out = legalbert(**enc)

        # Mean pool over non-padding tokens
        mask     = enc["attention_mask"].unsqueeze(-1).float()
        mean_vec = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        all_embeddings.append(mean_vec.cpu().numpy())

        if (i // batch_size) % 5 == 0:
            print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} cases...")

    return np.vstack(all_embeddings)   # (n_cases, 768)

print("Extracting facts embeddings...")
facts_embeddings = get_mean_embeddings(df["facts_text"].tolist())   # (n, 768)

print("Extracting issue embeddings...")
issue_embeddings = get_mean_embeddings(df["issue_text"].tolist())   # (n, 768)

print(f"Facts embeddings:  {facts_embeddings.shape}")
print(f"Issue embeddings:  {issue_embeddings.shape}")

Extracting facts embeddings...


  Embedded 16/424 cases...
  Embedded 96/424 cases...
  Embedded 176/424 cases...
  Embedded 256/424 cases...
  Embedded 336/424 cases...
  Embedded 416/424 cases...
Extracting issue embeddings...
  Embedded 16/424 cases...
  Embedded 96/424 cases...
  Embedded 176/424 cases...
  Embedded 256/424 cases...
  Embedded 336/424 cases...
  Embedded 416/424 cases...
Facts embeddings:  (424, 768)
Issue embeddings:  (424, 768)


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. LATE FUSION
#    Concatenate [facts(768) | issue(768) | tribunal(16)]  →  1552-dim
# ─────────────────────────────────────────────────────────────────────────────

X_all = np.hstack([facts_embeddings, issue_embeddings, tribunal_vecs])
print(f"Fused feature matrix: {X_all.shape}")   # (n_cases, 1552)

Fused feature matrix: (424, 1552)


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. LABELS
# ─────────────────────────────────────────────────────────────────────────────

label_map = {'Not Liable': 0, 'Liable': 1}
y_all = df["defendant_label"].map(label_map).values

print("Label mapping:")
print("  0 = Not Liable")
print("  1 = Liable")
unique, counts = np.unique(y_all, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  {val} ({['Not Liable','Liable'][val]}): {count}")

Label mapping:
  0 = Not Liable
  1 = Liable
  0 (Not Liable): 213
  1 (Liable): 211


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. TRAIN / TEST SPLIT  (group-based to prevent data leakage)
# ─────────────────────────────────────────────────────────────────────────────

case_ids = df["Case_Number"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=case_ids))

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Unique train cases: {len(set(case_ids[train_idx]))}  |  "
      f"Unique test cases: {len(set(case_ids[test_idx]))}")

Train: (347, 1552)  |  Test: (77, 1552)
Unique train cases: 109  |  Unique test cases: 28


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. GRID SEARCH  (raw LinearSVC — no calibration wrapper)
# ─────────────────────────────────────────────────────────────────────────────

param_grid = {
    'C':        [0.01, 0.1, 1.0, 10.0],
    'max_iter': [10000, 20000, 30000],
    'loss':     ['squared_hinge', 'hinge']
}

grid_search = GridSearchCV(
    LinearSVC(class_weight="balanced", random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

print("Starting GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters:       {grid_search.best_params_}")
print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")

results_df = pd.DataFrame(grid_search.cv_results_)
print("\nTop 5 parameter combinations:")
print(results_df[['param_C', 'param_max_iter', 'param_loss', 'mean_test_score']]
      .sort_values('mean_test_score', ascending=False)
      .head())


Starting GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters:       {'C': 10.0, 'loss': 'hinge', 'max_iter': 10000}
Best CV F1 (weighted): 0.4947

Top 5 parameter combinations:
    param_C  param_max_iter     param_loss  mean_test_score
21     10.0           10000          hinge         0.494750
22     10.0           20000          hinge         0.494750
23     10.0           30000          hinge         0.494750
19     10.0           20000  squared_hinge         0.491926
18     10.0           10000  squared_hinge         0.491926


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.5 CALIBRATE BEST MODEL
#      Fit CalibratedClassifierCV on full X_train using best params
# ─────────────────────────────────────────────────────────────────────────────

best_svm = LinearSVC(
    **grid_search.best_params_,
    class_weight="balanced",
    random_state=42
)

best_model = CalibratedClassifierCV(best_svm, cv=5)
best_model.fit(X_train, y_train)
print("Calibrated best model fitted.")

Calibrated best model fitted.


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.5 GRID SEARCH FOR OPTIMAL HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

base_svm_grid = LinearSVC(class_weight="balanced", random_state=42)

param_grid = {
    'C':        [0.01, 0.1, 1.0, 10.0],
    'max_iter': [20000, 30000],
    'loss':     ['squared_hinge', 'hinge']
}

grid_search = GridSearchCV(
    base_svm_grid,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train, y_train)
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1:  {grid_search.best_score_:.4f}")

# Calibrate AFTER finding best params, fit on full X_train
best_svm = LinearSVC(**grid_search.best_params_, class_weight="balanced", random_state=42)
best_model = CalibratedClassifierCV(best_svm, cv=5)
best_model.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best params: {'C': 10.0, 'loss': 'hinge', 'max_iter': 20000}
Best CV F1:  0.4947


,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",LinearSVC(C=1...ndom_state=42)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'hinge'
,"dual dual: ""auto"" or bool, de

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

label_names  = {0: 'Not Liable', 1: 'Liable'}
target_names = [label_names[0], label_names[1]]

print(classification_report(y_test, y_pred, target_names=target_names))

proba_df = pd.DataFrame(y_proba, columns=[f"P({name})" for name in target_names])
proba_df["Predicted"]  = [label_names[pred] for pred in y_pred]
proba_df["True_Label"] = [label_names[true] for true in y_test]
print("\nPrediction probabilities:")
print(proba_df.head(10))

              precision    recall  f1-score   support

  Not Liable       0.38      0.07      0.12        43
      Liable       0.42      0.85      0.56        34

    accuracy                           0.42        77
   macro avg       0.40      0.46      0.34        77
weighted avg       0.39      0.42      0.31        77


Prediction probabilities:
   P(Not Liable)  P(Liable)   Predicted  True_Label
0       0.448911   0.551089      Liable      Liable
1       0.484776   0.515224      Liable      Liable
2       0.497404   0.502596      Liable      Liable
3       0.520635   0.479365  Not Liable      Liable
4       0.497202   0.502798      Liable  Not Liable
5       0.507548   0.492452  Not Liable  Not Liable
6       0.467494   0.532506      Liable      Liable
7       0.474585   0.525415      Liable      Liable
8       0.490385   0.509615      Liable  Not Liable
9       0.493637   0.506363      Liable  Not Liable
